## Analysing snippets

In [ ]:
import pandas as pd
from driver import clean_text, get_text_stats
import numpy as np
import re

results_navigation = pd.read_csv("Data/results_navigation.csv")
results_navigation["timestamp"] = pd.to_datetime(results_navigation["timestamp"])

views_df = pd.read_csv("Data/viewed_results_df.csv")
views_df = views_df.loc[views_df["viewType"]=="resultExposureEnded"]

results_navigation["snippet"] = results_navigation["snippet"].apply(lambda x: clean_text(x))
# results_navigation["snippet"] = results_navigation["snippet"].apply(lambda x: re.sub(
#     r'^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\s+\d{1,2},\s+\d{4}\s*(?:\.\.\.)?\s*',
#     '',
#     x,
#     flags=re.IGNORECASE
# ))
results_navigation["snippet"] = results_navigation["snippet"].apply(lambda x: x.strip())
results_navigation["SERP_result"] = [str(title).strip()+snippet for title, snippet in zip(results_navigation["title"], results_navigation["snippet"])]

text_stat_results = get_text_stats(results_navigation, "SERP_result")

results_navigation["SERP_result_content_len"] = text_stat_results["content_len"]
results_navigation["SERP_result_mean_aoa"] = text_stat_results["mean_aoa"]
results_navigation["SERP_result_mean_conc"] = text_stat_results["mean_conc"]
results_navigation["SERP_result_gulpease_index"] = text_stat_results["gulpease_index"]
# results_navigation["snippet_fres"] = text_stat_results["fres"]

view_count = []
for _, row in results_navigation.iterrows():
    views = views_df.loc[(views_df["serp_id"]==row["serp_id"])&(views_df["rank"]==row["rank"])]
    view_count.append(len(views))
results_navigation["view_count"] = view_count
results_navigation.to_csv("Data/results_navigation.csv", index=False, encoding="utf8")

In [ ]:
results_navigation.to_csv("Data/results_navigation.csv", index=False, encoding='utf8')

In [ ]:
results_navigation.loc[pd.isna(results_navigation["rank"])]

## Analysing navigation

In [ ]:
import pandas as pd

# nlp = spacy.load("it_core_news_lg")
navigation_quality = pd.read_csv("Data/navigation_quality.csv")
navigation_quality["last_relevant_timestamp"] = pd.to_datetime(navigation_quality["last_relevant_timestamp"])

serp_df = pd.read_csv("Data/results_navigation.csv")
serp_df["timestamp"] = pd.to_datetime(serp_df["timestamp"])

query_df = pd.read_csv("Data/query_df.csv")
query_df["timestamp"] = pd.to_datetime(query_df["timestamp"])

click_df = pd.read_csv("Data/clicked_results_df.csv")
click_df["timestamp"] = pd.to_datetime(click_df["timestamp"])

view_df = pd.read_csv("Data/viewed_results_df.csv")
view_df["timestamp"] = pd.to_datetime(view_df["timestamp"])
view_df["associated_click_timestamp"] = pd.to_datetime(view_df["associated_click_timestamp"])

In [ ]:
serp_df.head()

In [ ]:
navigation_quality["serp_id"].unique()

In [ ]:
view_df["serp_id"].unique()

In [ ]:
serp_id = "serp78"
viewed_results = view_df.loc[(view_df["serp_id"]==serp_id) & (view_df["viewType"]=="resultExposureEnded")]
serp_results = serp_df.loc[serp_df["serp_id"]==serp_id]

unique_viewed_results = serp_results.loc[serp_results["url"].isin(viewed_results["url"].unique())]
unique_viewed_results

In [ ]:
import numpy as np

first_click_time = [] # time from query entry to first click
click_counts = [] # number of clicks made on a SERP
avg_scroll_time_before_click = [] # how much time is spent scrolling on SERP before a click
click_depth = [] # rank of the lowest clicked result
# avg_snippet_dwell_time = [] # average time for which a snippet is viewed at a time 
rank_lowest_view = [] # rank of the lowest viewed result
avg_snippet_view_counts_before_click = [] # average number of times a user views a snippet before making a click
avg_snippet_dwell_time_before_click = [] # average time a user dwells on a snippet at a time prior to making a click

# average text stats of all SERP results
SERP_total_results = []
SERP_highest_rel_result = []
SERP_avg_aoa = []
SERP_avg_conc = []
SERP_avg_length = []
SERP_avg_gulpease_index = []

# average text stats of all non-viewed results
no_view_avg_aoa = []
no_view_avg_conc = []
no_view_avg_length = []
no_view_avg_gulpease_index = []
no_view_results_ratio = [] # what ratio of serp results was never viewed


# average text stats of all viewed results
viewed_avg_aoa = []
viewed_avg_conc = []
viewed_avg_length = []
viewed_avg_gulpease_index = []

for serp_id in navigation_quality["serp_id"]:
    query_temp_df = query_df.loc[query_df["serp_id"] == serp_id] # query that yielded the SERP
    click_temp_df = click_df.loc[click_df["serp_id"] == serp_id] # all the clicks done on this SERP
    serp_temp_df = serp_df.loc[serp_df["serp_id"]==serp_id] # all the SERP results
    view_temp_df = view_df.loc[(view_df["serp_id"]==serp_id) & (view_df["viewType"]=="resultExposureEnded")] # all the snippets viewed in a SERP

    rank_lowest_view.append(view_temp_df["rank"].max()) # the last ranked result viewed

    if len(click_temp_df)==0: # no clicks were made on this SERP
        first_click_time.append(None) 
        avg_scroll_time_before_click.append(None)
        avg_snippet_dwell_time_before_click.append(None)
        avg_snippet_view_counts_before_click.append(None)
        click_counts.append(0)
        click_depth.append(None)
    else:
        first_click_timestamp = click_temp_df["timestamp"].min()
        last_serp_gen_timestamp = serp_temp_df.loc[serp_temp_df["timestamp"] <= first_click_timestamp]["timestamp"].max()
        first_click_time.append((first_click_timestamp - last_serp_gen_timestamp).total_seconds()) # duration between query submission and first click
        click_counts.append(len(click_temp_df["url"].unique())) # number of clicks made on this SERP
        click_depth.append(max(click_temp_df["rank"]))
        if len(view_temp_df) == 0: # user did not view any results
            avg_scroll_time_before_click.append(None)
            avg_snippet_dwell_time_before_click.append(None)
            avg_snippet_view_counts_before_click.append(None)
        else:
            scroll_time_before_click = [] # duration of time spent scrolling on the SERP before clicking on a result
            snippet_dwell_time_before_scroll = [] # duration of time spent looking at a snippet at a time before clicking on a SERP result
            avg_snippet_view_counts = [] # number of times a snippet is viewed before a click

            for _, click in click_temp_df.iterrows():
                views_before_click = view_temp_df.loc[view_temp_df["associated_click_timestamp"]==click["timestamp"]] # all snippets viewed before a click was made
                snippet_view_time = {rank: 0 for rank in views_before_click["rank"].unique()} # dictionary to store the time spent viewing each snippet before a click was made

                for _, view in views_before_click.iterrows():
                    snippet_view_time[view["rank"]] += view["viewDurationInSeconds"] # store the time spent viewing each snippet before a click was made

                avg_snippet_view_time = np.mean(list(snippet_view_time.values()))

                scroll_time_before_click.append((click["timestamp"] - views_before_click["timestamp"].min()).total_seconds())
                # snippet_dwell_time_before_scroll.append(np.mean(views_before_click["viewDurationInSeconds"].dropna()))
                snippet_dwell_time_before_scroll.append(avg_snippet_view_time)
                snippet_dwell_time_before_scroll.append(np.mean(views_before_click["viewDurationInSeconds"].dropna()))
                avg_snippet_view_counts.append(views_before_click["rank"].value_counts().mean())

            avg_scroll_time_before_click.append(np.mean(scroll_time_before_click))
            avg_snippet_dwell_time_before_click.append(np.mean(snippet_dwell_time_before_scroll))
            avg_snippet_view_counts_before_click.append(np.mean(avg_snippet_view_counts))

    # if len(click_temp_df) <= 1:
    #     avg_dwell_time_before_click.append(0)
    # else:
    #     click_temp_df = click_df.loc[click_df["serp_id"] == serp_id]

    #     for i, clicks in click_temp_df[1:].iterrows():
    #         time_before_click.append((click_temp_df.iloc[i]["timestamp"]- click_temp_df.iloc[i-1]["timestamp"]).total_seconds())
    #     avg_dwell_time_before_click.append(np.mean(time_before_click))

    no_view_SERP = serp_temp_df[~serp_temp_df["rank"].isin(view_temp_df["rank"].unique())]
    no_view_results_ratio.append(len(no_view_SERP["rank"].unique())/len(serp_temp_df["rank"].unique()))
    no_view_avg_aoa.append(no_view_SERP["SERP_result_mean_aoa"].dropna().mean())
    no_view_avg_conc.append(no_view_SERP["SERP_result_mean_conc"].dropna().mean())
    no_view_avg_gulpease_index.append(no_view_SERP["SERP_result_gulpease_index"].dropna().mean())
    no_view_avg_length.append(no_view_SERP["SERP_result_content_len"].dropna().mean())

    SERP_total_results.append(len(serp_temp_df["rank"].unique()))
    SERP_highest_rel_result.append(serp_temp_df.loc[serp_temp_df["relevance_score"]==1]["rank"].min())
    SERP_avg_aoa.append(serp_temp_df["SERP_result_mean_aoa"].dropna().mean())
    SERP_avg_conc.append(serp_temp_df["SERP_result_mean_conc"].dropna().mean())
    SERP_avg_gulpease_index.append(serp_temp_df["SERP_result_gulpease_index"].dropna().mean())
    SERP_avg_length.append(serp_temp_df["SERP_result_content_len"].dropna().mean())

    if len(view_temp_df) == 0:
        viewed_avg_aoa.append(None)
        viewed_avg_conc.append(None)
        viewed_avg_length.append(None)
        viewed_avg_gulpease_index.append(None)
    else:
        unique_viewed_results = serp_temp_df.loc[serp_temp_df["url"].isin(view_temp_df["url"].unique())]
        viewed_avg_aoa.append(unique_viewed_results["SERP_result_mean_aoa"].dropna().mean())
        viewed_avg_conc.append(unique_viewed_results["SERP_result_mean_conc"].dropna().mean())
        viewed_avg_length.append(unique_viewed_results["SERP_result_gulpease_index"].dropna().mean())
        viewed_avg_gulpease_index.append(unique_viewed_results["SERP_result_content_len"].dropna().mean())

navigation_quality["first_click_time"] = first_click_time
navigation_quality["total_clicks"] = click_counts
navigation_quality["click_depth"] = click_depth
navigation_quality["avg_scroll_time_before_click"] = avg_scroll_time_before_click
navigation_quality["avg_snippet_dwell_time_before_click"] = avg_snippet_dwell_time_before_click
navigation_quality["avg_snippet_view_counts_before_click"] = avg_snippet_view_counts_before_click
navigation_quality["inspect_depth"] = rank_lowest_view


navigation_quality["viewed_avg_aoa"] = viewed_avg_aoa
navigation_quality["viewed_avg_conc"] = viewed_avg_conc
navigation_quality["viewed_avg_length"] = viewed_avg_length
navigation_quality["viewed_avg_gulpease_index"] = viewed_avg_gulpease_index

navigation_quality["no_view_avg_aoa"] = no_view_avg_aoa
navigation_quality["no_view_avg_conc"] = no_view_avg_conc
navigation_quality["no_view_avg_length"] = no_view_avg_length
navigation_quality["no_view_avg_gulpease_index"] = no_view_avg_gulpease_index
navigation_quality["no_view_results_ratio"] = no_view_results_ratio

navigation_quality["SERP_total_results"] = SERP_total_results   
navigation_quality["SERP_highest_rel_result"] = SERP_highest_rel_result
navigation_quality["SERP_avg_aoa"] = SERP_avg_aoa
navigation_quality["SERP_avg_conc"] = SERP_avg_conc
navigation_quality["SERP_avg_gulpease_index"] = SERP_avg_gulpease_index
navigation_quality["SERP_avg_length"] = SERP_avg_length

navigation_quality["last_relevant_timestamp"] = pd.to_datetime(navigation_quality["last_relevant_timestamp"])
navigation_quality.to_csv("Data/navigation_quality.csv", index=False, encoding="utf8")

In [ ]:
navigation_quality.loc[navigation_quality["serp_id"]=="serp35"]

In [ ]:
navigation_quality.head()

In [ ]:
navigation_quality.loc[navigation_quality["serp_id"]=="serp35"]

## Visualisation

In [ ]:
import pandas as pd

navigation_quality = pd.read_csv("Data/navigation_quality.csv")
categories = [
    "inspect_depth", "click_depth", "no_view_results_ratio", "total_clicks", "repetitions",
    "first_click_time", "avg_scroll_time_before_click", 
    # "avg_snippet_dwell_time_before_click", 
    "avg_snippet_view_counts_before_click", 
    # "tail", 
    # "SERP_total_results", 
    
    # "SERP_highest_rel_result",
    "viewed_avg_length", #"SERP_avg_length", 
    "viewed_avg_gulpease_index", #"SERP_avg_gulpease_index", 
    "viewed_avg_conc", #"SERP_avg_conc", 
    "viewed_avg_aoa", #"SERP_avg_aoa", 
    
]
# navigation_quality

In [ ]:
print(len(categories))

In [ ]:
navigation_quality.loc[navigation_quality["no_view_results_ratio"]==1]

### RQ1

In [ ]:
from driver import get_descriptive_stats
import pandas as pd

navigation_quality = pd.read_csv("Data/navigation_quality.csv")

binwidth_map = {
    "first_click_time": 50,
    "avg_scroll_time_before_click": 50,
    "avg_snippet_dwell_time_before_click": 10,
    "avg_snippet_view_counts_before_click": 0.1,
    "inspect_depth": 1,
    "viewed_avg_length": 5,
    "viewed_avg_gulpease_index": 1,
    "viewed_avg_conc": 0.5,
    "viewed_avg_aoa": 0.5,
    "SERP_avg_length": 5,
    "SERP_avg_gulpease_index": 1,
    "SERP_avg_conc": 0.5,
    "SERP_avg_aoa": 0.5,
    "no_view_results_ratio": 0.1,
    "repetitions": 1,
    "tail": 1,
    "SERP_total_results": 1,
    "SERP_highest_rel_result": 1,
    "total_clicks": 1,
    "click_depth":1
}

desc_stats = get_descriptive_stats(df=navigation_quality, search_stage="results_navigation", categories=categories, ncols=3, nrows=4, binwidth_map=binwidth_map)
desc_stats

In [ ]:
results_navigation = pd.read_csv("Data/results_navigation.csv")
len(results_navigation.loc[results_navigation["click_count"]>1]), len(results_navigation.loc[results_navigation["click_count"]==1]), len(results_navigation)

In [ ]:
import pandas as pd
navigation_quality = pd.read_csv("Data/navigation_quality.csv")
len(navigation_quality.loc[navigation_quality["total_clicks"]==0])/len(navigation_quality)

In [ ]:
import pandas as pd
navigation_quality = pd.read_csv("Data/navigation_quality.csv")
len(navigation_quality.loc[navigation_quality["total_clicks"]==0])/len(navigation_quality)

In [ ]:
len(navigation_quality.loc[navigation_quality["first_click_time"]<=60])/len(navigation_quality.loc[navigation_quality["total_clicks"]>0])

In [ ]:
len(navigation_quality.loc[navigation_quality["avg_scroll_time_before_click"]<=18])/len(navigation_quality.loc[navigation_quality["total_clicks"]>0])


In [ ]:
len(navigation_quality.loc[navigation_quality["avg_scroll_time_before_click"]<=50])/len(navigation_quality[~pd.isna(navigation_quality["avg_scroll_time_before_click"])])

In [ ]:
len(navigation_quality.loc[navigation_quality["avg_snippet_dwell_time_before_click"]<=15])/len(navigation_quality.loc[navigation_quality["total_clicks"]>0])

In [ ]:
len(navigation_quality.loc[navigation_quality["rank_lowest_view"]<=5])/len(navigation_quality)


In [ ]:
len(navigation_quality.loc[navigation_quality["rank_lowest_view"]<=10])/len(navigation_quality)

In [ ]:
len(navigation_quality.loc[navigation_quality["no_view_results_ratio"]>=0.8])/len(navigation_quality)

In [ ]:
len(navigation_quality.loc[navigation_quality["SERP_avg_gulpease_index"]>=80])/len(navigation_quality)

In [ ]:
results_navigation = pd.read_csv("Data/results_navigation.csv")
len(results_navigation.loc[results_navigation["SERP_result_gulpease_index"]>=80])/len(results_navigation)

In [ ]:
len(navigation_quality)

In [ ]:
from driver import get_descriptive_stats

extra_categories = [
    # "first_click_time", "avg_scroll_time_before_click", "avg_snippet_dwell_time_before_click", "avg_snippet_view_counts_before_click", "rank_lowest_view", 
    # "repetitions", "tail", 
    # "SERP_total_results", "SERP_highest_rel_result",
    "viewed_avg_length", 
    "viewed_avg_gulpease_index", 
    "viewed_avg_conc", 
    "viewed_avg_aoa", 
    "SERP_avg_length", "SERP_avg_gulpease_index", "SERP_avg_conc", "SERP_avg_aoa"
    # "no_view_results_ratio"
]

binwidth_map = {
    "first_click_time": 50,
    "avg_scroll_time_before_click": 50,
    "avg_snippet_dwell_time_before_click": 10,
    "avg_snippet_view_counts_before_click": 0.1,
    "rank_lowest_view": 1,
    "viewed_avg_length": 5,
    "viewed_avg_gulpease_index": 1,
    "viewed_avg_conc": 0.5,
    "viewed_avg_aoa": 0.5,
    "SERP_avg_length": 5,
    "SERP_avg_gulpease_index": 1,
    "SERP_avg_conc": 0.5,
    "SERP_avg_aoa": 0.5,
    "no_view_results_ratio": 0.1,
    "repetitions": 1,
    "tail": 1,
    "click_depth": 1,
    "SERP_total_results": 1,
    "SERP_highest_rel_result": 1
}

desc_stats = get_descriptive_stats(df=navigation_quality.loc[navigation_quality["no_view_results_ratio"]<1], search_stage="results_navigation_extra", categories=extra_categories, ncols=4, nrows=2, binwidth_map=binwidth_map)
desc_stats

In [ ]:
results_navigation.head()

In [ ]:
len(navigation_quality.loc[navigation_quality["no_view_results_ratio"]==1])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

navigation_quality = pd.read_csv("Data/navigation_quality.csv")
df = navigation_quality.loc[navigation_quality["no_view_results_ratio"]<1] # children viewed at least one result
# results_navigation = pd.read_csv("Data/results_navigation.csv")
# viewed_results = results_navigation.loc[results_navigation["view_count"]>0]
# not_viewed_results = results_navigation.loc[results_navigation["view_count"]==0]
# print(len(df))
binwidth_map = {
    "viewed_avg_length": 5,
    "viewed_avg_gulpease_index": 1,
    "viewed_avg_conc": 0.5,
    "viewed_avg_aoa": 0.5,
    "SERP_avg_length": 5,
    "SERP_avg_gulpease_index": 1,
    "SERP_avg_conc": 0.5,
    "SERP_avg_aoa": 0.5,
}
plt.figure(figsize=(16,9))
plt.subplots_adjust(hspace=0.25)
ncols = 2
nrows = 2
for i, category in enumerate(["avg_length", "avg_gulpease_index", "avg_conc", "avg_aoa"]):
# for i, category in enumerate(["SERP_result_content_len", "SERP_result_mean_aoa", "SERP_result_mean_conc", "SERP_result_gulpease_index"]):
    ax = plt.subplot(nrows, ncols, i+1)
    viewed = df["viewed_"+category]
    not_viewed = df["no_view_"+category]
    SERP = df["SERP_"+category]
    sns.histplot(
            data=viewed,
            stat="proportion",
            ax = ax,
            alpha=0.1,
            kde=True,
            kde_kws={"cut":0},
            color="green",
            label="Viewed results"
        )
    viewed_median = viewed.median()
    ax.axvline(viewed_median, linestyle="--", color="green", label="median value for viewed results")

    # sns.histplot(
    #         data=not_viewed,
    #         stat="proportion",
    #         ax = ax,
    #         alpha=0.1,
    #         kde=True,
    #         kde_kws={"cut":0},
    #         color="green",
    #         label="avg of not viewed results"
    #     )
    # not_viewed_median = not_viewed.median()
    # ax.axvline(not_viewed_median, linestyle="--", color="green", label="not_viewed_median")

    sns.histplot(
            data=SERP,
            stat="proportion",
            ax=ax,
            alpha=0.1,
            kde=True,
            kde_kws={"cut":0},
            color="orange",
            label="All SERP results"
        )
    SERP_median = SERP.median()
    ax.axvline(SERP_median, linestyle="--", color="orange", label="median value for all SERP results")
    ax.set_xlabel(category)
# handles, labels = ax.get_legend_handles_labels()
# plt.figlegend(handles, labels, loc="upper center", ncol=6, fontsize=10)
plt.savefig("Figures/viewed_vs_SERP_results.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
results_navigation = pd.read_csv("Data/results_navigation.csv")
results_navigation.head()
# rank_wise = results_navigation.groupby("rank")[["SERP_result_content_len", "SERP_result_mean_aoa", "SERP_result_mean_conc", "SERP_result_gulpease_index"]].mean().reset_index(drop=False)
# nrows = 2
# ncols = 2
# plt.figure(figsize=(16,9))
# plt.subplots_adjust(hspace=0.25)
# for i, category in enumerate(["SERP_result_content_len", "SERP_result_mean_aoa", "SERP_result_mean_conc", "SERP_result_gulpease_index"]):
#     ax = plt.subplot(nrows, ncols, i+1)
#     sns.barplot(
#             data=rank_wise,
#             x = "rank",
#             y = category,
#             ax = ax,
#         )

### RQ2

In [ ]:
len(navigation_quality), len(navigation_quality.loc[navigation_quality["effective_navigation"]==1])

In [ ]:
from driver import get_significance

categories = [
    "inspect_depth", "click_depth", "no_view_results_ratio", "total_clicks", "repetitions",
    "first_click_time", "avg_scroll_time_before_click", 
    # "avg_snippet_dwell_time_before_click", 
    "avg_snippet_view_counts_before_click", 
    # "tail", 
    # "SERP_total_results", 
    
    # "SERP_highest_rel_result",
    "viewed_avg_length", #"SERP_avg_length", 
    "viewed_avg_gulpease_index", #"SERP_avg_gulpease_index", 
    "viewed_avg_conc", #"SERP_avg_conc", 
    "viewed_avg_aoa", #"SERP_avg_aoa", 
    
]

desc_stats_df_tail_ratio, sig_test_results_tail_ratio = get_significance(df=navigation_quality, effective_label="effective_navigation", search_stage="results_navigation", categories=categories)
sig_test_results_tail_ratio

In [ ]:
desc_stats_df_tail_ratio

In [ ]:
from driver import get_boxplots

get_boxplots(df=navigation_quality, search_stage="results_navigation", categories=categories, group="effective_navigation", artefact="navigation", n_cols=3, n_rows=4, stats_df=sig_test_results_tail_ratio)